# Chemprop User Guide

**Version:** 1.7.1

**Audience:** Scientists and HPC users training molecular property prediction models with command-line workflows

## TL;DR

Chemprop is a message passing neural network toolkit for molecular property prediction. In this HPC environment, you use it as a command-line application via modules.

Quick start:

```bash
module load chemprop/1.7.1
chemprop_train --version
chemprop_train --help
```

This notebook focuses on reproducible CLI usage, SLURM-ready workflows, and troubleshooting on shared systems.

## Table of Contents

- [TL;DR](#tldr)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute QuickStart](#tldr-5-minute-quickstart)
  - [Step 1: Load the Module](#step-1-load-the-module)
  - [Step 2: Verify Setup](#step-2-verify-setup)
  - [Step 3: Run Your First Job](#step-3-run-your-first-job)
- [PART II — How to Use Chemprop (Read Carefully Once)](#part-ii--how-to-use-chemprop-read-carefully-once)
  - [What You Need to Know](#what-you-need-to-know)
  - [CPU vs GPU: When to Use Each](#cpu-vs-gpu-when-to-use-each)
  - [Keyboard Shortcuts / Options](#keyboard-shortcuts--options)
  - [Resource Requests](#resource-requests)
- [PART III — Workflows (Use As Needed)](#part-iii--workflows-use-as-needed)
  - [Workflow 1: Train a Regression Model](#workflow-1-train-a-regression-model)
  - [Workflow 2: Predict from a Saved Checkpoint](#workflow-2-predict-from-a-saved-checkpoint)
  - [Workflow 3: Generate Molecular Fingerprints](#workflow-3-generate-molecular-fingerprints)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Symptom-Based Lookup](#symptom-based-lookup)
  - [FAQ](#faq)
  - [External Resources](#external-resources)

## PART I — Getting Started (Read Once)

### TL;DR: 5-Minute QuickStart

```bash
module load chemprop/1.7.1
chemprop_train --version
chemprop_train --help
```

### Step 1: Load the Module

In [ ]:
%%bash
module avail chemprop 2>&1 | head -40

**Expected output:**
```text
One or more Chemprop modules are listed (for example chemprop/1.7.1).
```

In [ ]:
%%bash
module load chemprop/1.7.1
module list 2>&1 | head -40

**Expected output:**
```text
The loaded module list includes chemprop/1.7.1.
```

### Step 2: Verify Setup

In [ ]:
%%bash
module load chemprop/1.7.1
chemprop_train --help 2>&1 | head -30

**Expected output:**
```text
Usage text for chemprop_train appears with available options.
```

Success! The command-line interface is available in your environment.

### Step 3: Run Your First Job

In [ ]:
%%bash
module load chemprop/1.7.1
for cmd in chemprop_train chemprop_predict chemprop_hyperopt chemprop_interpret chemprop_fingerprint chemprop_web; do
  echo "===== ${cmd} --help ====="
  ${cmd} --help > ${cmd}_help.txt 2>&1 || true
  head -20 ${cmd}_help.txt
  echo
done

**Expected output:**
```text
Help summaries are printed for each Chemprop CLI command, and full help text is saved to *_help.txt files.
```

Your first job is complete: you verified all major Chemprop entry points and captured full option lists.

## PART II — How to Use Chemprop (Read Carefully Once)

### What You Need to Know

- Chemprop accepts molecular data in CSV format (SMILES plus targets).
- Common task types: regression, classification, multiclass, spectra.
- Primary workflow is train first, then predict using saved checkpoints.
- In this environment, use module commands for setup and CLI commands for execution.

| Platform | Status | Notes |
|---|---|---|
| CPU-only nodes | Supported | Good for small datasets and debugging |
| GPU nodes | Supported | Recommended for large datasets and ensembles |
| Login node | Limited use | Use for setup, help, and lightweight checks only |

### CPU vs GPU: When to Use Each

Use CPU when:
- you are validating data format
- you are testing command syntax
- your dataset is small

Use GPU when:
- training deep ensembles
- using large datasets
- time-to-result matters

Decision guide:

```text
Prototype or small data?
  yes -> CPU
  no  -> GPU
```

### Keyboard Shortcuts / Options

Chemprop is non-interactive CLI software, so focus on options instead of keyboard shortcuts.

Key command families:
- `chemprop_train`: model training and cross-validation
- `chemprop_predict`: batch inference with saved checkpoints
- `chemprop_hyperopt`: hyperparameter optimization
- `chemprop_interpret`: model interpretation tools
- `chemprop_fingerprint`: latent/fingerprint extraction

To inspect full flags for any command:

```bash
module load chemprop/1.7.1
chemprop_train --help
```

### Resource Requests

Chemprop training is compute-intensive. Use scheduler allocations for real training jobs.

Example SLURM template:

```bash
#!/bin/bash
#SBATCH --job-name=chemprop-train
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=04:00:00

module load chemprop/1.7.1
chemprop_train --data_path data.csv --dataset_type regression --save_dir checkpoints
```

For help-only and metadata checks, no allocation is needed on most clusters.

## PART III — Workflows (Use As Needed)

**Disclaimer (sequential workflow behavior):**

- Run Workflow 1 first. It creates the synthetic input data and baseline training artifacts.
- Workflows 2 and 3 are sequential and depend on files produced by Workflow 1.
- All three workflows use the same working directory: `./workflow`.
- Each downstream workflow performs file-existence checks before submitting jobs and exits early with a clear message if prerequisites are missing.


### Workflow 1: Train a Regression Model

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow
rm -rf "${WORKDIR}"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow1.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf1
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow1.out

set -euo pipefail

module load chemprop/1.7.1

echo "Creating synthetic regression dataset..."
cat > regression_train.csv << 'CSV'
smiles,target
CC,0.10
CCC,0.20
CCCC,0.30
CCCCC,0.40
CCCCCC,0.50
C=O,0.60
CC=O,0.70
CCC=O,0.80
CCO,0.90
CCCO,1.00
CCCCO,1.10
CCN,1.20
CCCN,1.30
CCCCN,1.40
c1ccccc1,1.50
c1ccncc1,1.60
c1ccoc1,1.70
CCCl,1.80
CCCCl,1.90
CCBr,2.00
CCCBr,2.10
COC,2.20
CCOC,2.30
CCCOC,2.40
CC(=O)O,2.50
CC(=O)N,2.60
CCS,2.70
CCCS,2.80
CCF,2.90
CCCF,3.00
CSV

echo "Training chemprop model..."
chemprop_train \
  --data_path regression_train.csv \
  --dataset_type regression \
  --save_dir checkpoints_reg \
  --epochs 3 \
  --split_sizes 0.8 0.1 0.1 \
  --num_workers 0 \
  > train.log 2>&1

echo "Verifying concrete outputs..."
test -f checkpoints_reg/fold_0/model_0/model.pt
test -f checkpoints_reg/fold_0/test_smiles.csv

find checkpoints_reg -maxdepth 4 -type f | sort > output_manifest.txt

echo "Workflow 1 completed successfully."
echo "Artifacts:"
echo "- regression_train.csv"
echo "- train.log"
echo "- checkpoints_reg/fold_0/model_0/model.pt"
echo "- checkpoints_reg/fold_0/test_smiles.csv"
echo "- output_manifest.txt"
SBATCH

sbatch chemprop_workflow1.sbatch

**Expected output:**
```text
The command runs successfully and Chemprop creates training artifacts inside ./workflow.

Chemprop-generated outputs:
- checkpoints_reg/fold_0/model_0/model.pt (binary; listing only)
- checkpoints_reg/fold_0/test_smiles.csv

Readable file preview (representative lines):
- checkpoints_reg/fold_0/test_smiles.csv:
  - smiles
  - CCO
  - CCN

Note: exact test-set rows can vary with data splitting.
```

### Workflow 2: Predict from a Saved Checkpoint

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow

echo "Checking Workflow 1 prerequisites..."
test -f "${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt" || {
  echo "Missing prerequisite: ${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt"
  echo "Run Workflow 1 first."
  exit 1
}
test -f "${WORKDIR}/checkpoints_reg/fold_0/test_smiles.csv" || {
  echo "Missing prerequisite: ${WORKDIR}/checkpoints_reg/fold_0/test_smiles.csv"
  echo "Run Workflow 1 first."
  exit 1
}

mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow2.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf2
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow2.out

set -euo pipefail

module load chemprop/1.7.1

test -f checkpoints_reg/fold_0/model_0/model.pt
test -f checkpoints_reg/fold_0/test_smiles.csv

echo "Running chemprop_predict using Workflow 1 artifacts..."
chemprop_predict \
  --test_path checkpoints_reg/fold_0/test_smiles.csv \
  --preds_path preds_reg.csv \
  --checkpoint_dir checkpoints_reg \
  > predict.log 2>&1

test -f preds_reg.csv
test -s preds_reg.csv

echo "Prediction file preview:"
head -n 10 preds_reg.csv

echo "Workflow 2 completed successfully."
SBATCH

sbatch chemprop_workflow2.sbatch

**Expected output:**
```text
The command checks for Workflow 1 artifacts first and exits with a clear message if they are missing.
If prerequisites exist, it runs successfully and Chemprop writes predictions.

Chemprop-generated outputs:
- preds_reg.csv

Readable file preview (representative lines):
- preds_reg.csv:
  - smiles,prediction
  - CCO,0.91
  - CCN,1.18

Note: prediction values vary by run and model state.
```

### Workflow 3: Generate Molecular Fingerprints

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow

echo "Checking Workflow 1 prerequisites..."
test -f "${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt" || {
  echo "Missing prerequisite: ${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt"
  echo "Run Workflow 1 first."
  exit 1
}
test -f "${WORKDIR}/regression_train.csv" || {
  echo "Missing prerequisite: ${WORKDIR}/regression_train.csv"
  echo "Run Workflow 1 first."
  exit 1
}

mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow3.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf3
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow3.out

set -euo pipefail

module load chemprop/1.7.1

test -f checkpoints_reg/fold_0/model_0/model.pt
test -f regression_train.csv

echo "Running chemprop_fingerprint using Workflow 1 artifacts..."
chemprop_fingerprint \
  --test_path regression_train.csv \
  --preds_path fingerprint.csv \
  --checkpoint_dir checkpoints_reg \
  --fingerprint_type MPN \
  > fingerprint.log 2>&1

test -f fingerprint.csv
test -s fingerprint.csv

echo "Fingerprint file preview:"
head -n 5 fingerprint.csv

echo "Workflow 3 completed successfully."
SBATCH

sbatch chemprop_workflow3.sbatch

**Expected output:**
```text
The command checks for Workflow 1 artifacts first and exits with a clear message if they are missing.
If prerequisites exist, it runs successfully and Chemprop writes fingerprints.

Chemprop-generated outputs:
- fingerprint.csv

Readable file preview (representative lines):
- fingerprint.csv:
  - smiles,fp_0,fp_1,fp_2,...
  - CC,0.12,-0.03,0.77,...
  - CCC,0.18,-0.10,0.81,...

Note: fingerprint dimensionality and numeric values depend on model configuration.
```

## PART IV — Troubleshooting (Skim When Broken)

### Symptom-Based Lookup

| Symptom | Likely Cause | Fix |
|---|---|---|
| `chemprop_train: command not found` | Module not loaded | `module load chemprop/1.7.1` |
| `No module named chemprop` | Wrong Python environment | Reload module and open a new shell/notebook kernel |
| CUDA not detected | Running on CPU node or missing GPU allocation | Submit to GPU partition and request `--gres=gpu:1` |
| Out-of-memory during training | Batch/model too large | Reduce batch size, request more memory, or use gradient checkpointing |
| CSV parsing error | Invalid delimiter/header/SMILES field | Validate CSV header and data format against Chemprop docs |

### FAQ

**Q: Can I run Chemprop on a login node?**

A: Use login nodes for setup and help commands only. Use scheduler allocations for training and large predictions.

**Q: Which command should I start with?**

A: Start with `chemprop_train --help`, then run a short training job on a small dataset.

**Q: Where can I find end-to-end examples?**

A: See the local workshop and benchmark resources listed in External Resources below.

### External Resources

- Official repository: https://github.com/chemprop/chemprop
- Main publication: https://pubs.acs.org/doi/10.1021/acs.jcim.3c01250
- Documentation homepage: https://chemprop.readthedocs.io/